This notebook geocodes the NYCTheaters.csv and matches it to the TLC zones.

In [1]:
import pandas as pd

theaters = pd.read_csv("../data/broadway_data/NYCTheaters.csv")

theaters.head()


,Theater_ID,Theater,Address,Current Location Name
0,1,46th Street Theatre,226 West 46th Street,Richard Rodgers Theatre
1,2,46th Street Theatre,302 West 45th Street,NaN
2,2,Alvin Theatre,250 West 52nd Street,Neil Simon Theatre
3,3,Ambassador Theatre,219 West 49th Street,NaN
4,4,ANITA Playhouse,245 West 52nd Street,August Wilson Theatre


In [2]:
import requests

def geocode_address(address):
    '''
    Geocode an address using the US Census Bureau's geocoding service.
    '''
    url = "https://geocoding.geo.census.gov/geocoder/locations/onelineaddress"
    
    params = {
        "address": f"{address}, New York, NY",
        "benchmark": "Public_AR_Current",
        "format": "json"
    }
    
    response = requests.get(url, params=params)
    data = response.json()
    
    matches = data["result"]["addressMatches"]
    
    if matches:
        coordinates = matches[0]["coordinates"]
        return coordinates["x"], coordinates["y"]
    
    return None, None

Add the coordinates to the theaters data frame.

In [3]:
theaters[["longitude", "latitude"]] = theaters["Address"].apply(
    lambda x: pd.Series(geocode_address(x))
)

Check how many theaters were successfully geocoded before doing anything else with the coordinates.

In [4]:
n_total = len(theaters)
n_failed = theaters["latitude"].isna().sum()

print(f"Geocoding match rate: {n_total - n_failed}/{n_total} "
      f"({(n_total - n_failed) / n_total:.1%}) matched")

if n_failed > 0:
    print("\nTheaters that failed to geocode (will be excluded until resolved):")
    print(theaters.loc[theaters["latitude"].isna(), ["Theater", "Address"]])

Geocoding match rate: 59/60 (98.3%) matched

Theaters that failed to geocode (will be excluded until resolved):
   Theater Address
59     NaN     NaN


Drop rows with a missing address, and separately drop rows where geocoding itself failed.

In [5]:
shape_before = theaters.shape
theaters = theaters.dropna(subset=["Address"]).copy()
theaters = theaters.dropna(subset=["latitude", "longitude"]).copy()
shape_after = theaters.shape

print(f"Theaters shape before dropping missing address/geocode: {shape_before}")
print(f"Theaters shape after dropping missing address/geocode:  {shape_after}")

theaters.to_csv(
    "../data/broadway_data/NYCTheaters_geocoded.csv",
    index=False
)
theaters.head(5)

Theaters shape before dropping missing address/geocode: (60, 6)
Theaters shape after dropping missing address/geocode:  (59, 6)


,Theater_ID,Theater,Address,Current Location Name,longitude,latitude
0,1,46th Street Theatre,226 West 46th Street,Richard Rodgers Theatre,-73.986081,40.758945
1,2,46th Street Theatre,302 West 45th Street,NaN,-73.988598,40.759186
2,2,Alvin Theatre,250 West 52nd Street,Neil Simon Theatre,-73.984259,40.763082
3,3,Ambassador Theatre,219 West 49th Street,NaN,-73.984853,40.761005
4,4,ANITA Playhouse,245 West 52nd Street,August Wilson Theatre,-73.984052,40.763127


Notice that there are duplicates in the theater column caused by name changes (the first entry is 46th Street Theatre but has changed their name to Richard Rodgers Theatre). So theaters is standardised and duplicates are dropped.

In [6]:
# standardize addresses
theaters["Address_standardised"] = (
    theaters["Address"]
    .str.upper()
    .str.replace(".", "", regex=False)
    .str.replace("WEST", "W", regex=False)
    .str.replace("EAST", "E", regex=False)
    .str.replace("STREET", "ST", regex=False)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()  
)

current_theaters = (
    theaters
    .assign(
        Current_Theatre=theaters["Current Location Name"]
        .fillna(theaters["Theater"]) # if Current Location Name is missing, use Theater name (this means the name has not been changed or it uses the new name)
    )
    .dropna(subset=["Address_standardised"])
    .drop_duplicates(subset=["Address_standardised"])
    .copy()
)


current_theaters[
    ["Current_Theatre", "Address", "Address_standardised",
     "longitude", "latitude"]
]

current_theaters.to_csv(
    "../data/broadway_data/current_theaters.csv",
    index=False
)

Load the TLC taxi zones and create a GeoDataFrame from the theater data.

In [7]:
import geopandas as gpd

zones = gpd.read_file("../data/taxi_zones/taxi_zones.shp")

theaters_gdf = gpd.GeoDataFrame(
    current_theaters,
    geometry=gpd.points_from_xy(
        current_theaters["longitude"],
        current_theaters["latitude"]
    ),
    crs="EPSG:4326"
)

Put the theater points and taxi zones on the same map and find which taxi-zone polygon contains each theater.

In [8]:
theaters_gdf = theaters_gdf.to_crs(zones.crs) # convert theaters_gdf to the same CRS as zones

theaters_zones = gpd.sjoin(
    theaters_gdf,
    zones,
    how="left",
    predicate="within"
)

theaters_zones.head(5)

,Theater_ID,Theater,Address,Current Location Name,longitude,latitude,Address_standardised,Current_Theatre,geometry,index_right,OBJECTID,Shape_Leng,Shape_Area,zone,LocationID,borough
0,1,46th Street Theatre,226 West 46th Street,Richard Rodgers Theatre,-73.986081,40.758945,226 W 46TH ST,Richard Rodgers Theatre,POINT (988105.982 215781.006),229,230,0.031028,0.000056,Times Sq/Theatre District,230,Manhattan
1,2,46th Street Theatre,302 West 45th Street,NaN,-73.988598,40.759186,302 W 45TH ST,46th Street Theatre,POINT (987408.671 215868.736),47,48,0.043747,0.000094,Clinton East,48,Manhattan
2,2,Alvin Theatre,250 West 52nd Street,Neil Simon Theatre,-73.984259,40.763082,250 W 52ND ST,Neil Simon Theatre,POINT (988610.481 217288.653),229,230,0.031028,0.000056,Times Sq/Theatre District,230,Manhattan
3,3,Ambassador Theatre,219 West 49th Street,NaN,-73.984853,40.761005,219 W 49TH ST,Ambassador Theatre,POINT (988446.175 216531.882),229,230,0.031028,0.000056,Times Sq/Theatre District,230,Manhattan
4,4,ANITA Playhouse,245 West 52nd Street,August Wilson Theatre,-73.984052,40.763127,245 W 52ND ST,August Wilson Theatre,POINT (988667.892 217304.73),229,230,0.031028,0.000056,Times Sq/Theatre District,230,Manhattan


Check for theaters that fell outside every taxi zone polygon (e.g. a geocoded point sitting exactly on a zone boundary, or a few metres outside it due to geocoding imprecision) rather than silently losing them.

In [9]:
n_unmatched = theaters_zones["LocationID"].isna().sum()
print(f"Theaters unmatched to a taxi zone: {n_unmatched} / {len(theaters_zones)}")

if n_unmatched > 0:
    print(theaters_zones.loc[
        theaters_zones["LocationID"].isna(),
        ["Current_Theatre", "Address", "longitude", "latitude"]
    ])

Theaters unmatched to a taxi zone: 0 / 41


For any unmatched theaters, fall back to nearest-zone matching instead of dropping them outright, since a small geocoding offset can put a point just outside its true zone polygon.

In [10]:
unmatched_names = theaters_zones.loc[
    theaters_zones["LocationID"].isna(), "Current_Theatre"
].tolist()

if unmatched_names:
    print(f"zones CRS: {zones.crs}")
    fallback = gpd.sjoin_nearest(
        theaters_gdf[theaters_gdf["Current_Theatre"].isin(unmatched_names)],
        zones,
        how="left",
        max_distance=200  # adjusted based on the zones.crs units
    )
    n_recovered = fallback["LocationID"].notna().sum()
    print(f"Recovered {n_recovered} of {len(unmatched_names)} previously unmatched "
          f"theaters via nearest-zone fallback")

    # merge recovered matches back into theaters_zones
    theaters_zones = theaters_zones.set_index("Current_Theatre")
    fallback = fallback.set_index("Current_Theatre")
    theaters_zones.update(fallback[["LocationID"]])
    theaters_zones = theaters_zones.reset_index()
else:
    print("No unmatched theaters — nothing to recover.")

No unmatched theaters — nothing to recover.


Save final result:

In [11]:
theaters_zones.to_csv(
    "../data/broadway_data/theaters_zones.csv",
    index=False
)